# 6장. 생성 속도 재기: KV 캐시와 배치

위키백과 문단으로 작은 언어 모델을 학습시킨 다음, 글을 만드는 데 걸리는 시간을 첫 토큰까지의 시간과 토큰당 시간으로 나눠 잰다. KV 캐시를 직접 구현해 결과가 같은지 확인하고, 캐시의 메모리와 배치의 효과를 잰다.

**준비:** 한국어 위키백과 덤프의 첫 조각 `kowiki-latest-pages-articles1.xml-p1p82407.bz2`(4장과 같은 파일)를 왼쪽 파일 창에 올리고, 코랩에서는 `!pip install tokenizers`를 먼저 실행한다. 저자의 맥(CPU 스레드 2개)에서 끝까지 실행하는 데 14분쯤 걸렸다.

## 6.1 문단으로 학습시킨 모델과 생성 루프

In [ ]:
import bz2, re, math, time, os, sys, random, json, platform
from collections import Counter
from xml.etree.ElementTree import iterparse
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
os.environ["RAYON_NUM_THREADS"] = "2"
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
torch.set_num_threads(2)

pages = []
with bz2.open("kowiki-latest-pages-articles1.xml-p1p82407.bz2", "rb") as f:
    for _, el in iterparse(f, events=("end",)):
        if el.tag.endswith("}page"):
            if el.findtext("{*}ns") == "0" and el.find("{*}redirect") is None:
                pages.append((el.findtext("{*}title"), el.findtext("{*}revision/{*}text") or ""))
            el.clear()
print(len(pages))
print(sys.version.split()[0], torch.__version__, tokenizers.__version__, platform.machine(), torch.get_num_threads())

In [ ]:
# 4장의 clean()을 그대로 쓴다. 이번에는 문장으로 나누지 않고 문단을 이어 문서 하나를 글 하나로 만든다
def strip_nested(text, left, right):
    """가장 안쪽의 left ... right를 지우기를 더 지울 것이 없을 때까지 한다(틀 {{ }}, 표 {| |})."""
    inner = re.compile(re.escape(left) + r"(?:(?!" + re.escape(left) + r").)*?" + re.escape(right), re.S)
    while True:
        new = inner.sub("", text)
        if new == text:
            return new
        text = new

def clean(wiki):
    """위키 문법을 걷어 내고 문단(줄) 목록을 돌려준다."""
    t = re.sub(r"<!--.*?-->", "", wiki, flags=re.S)                       # 주석
    t = re.sub(r"<ref[^>]*/>", "", t)                                      # 각주
    t = re.sub(r"<ref[^>]*>.*?</ref>", "", t, flags=re.S)
    t = strip_nested(t, "{{", "}}")                                        # 틀(정보 상자 등)
    t = strip_nested(t, "{|", "|}")                                        # 표
    t = re.sub(r"<(math|gallery|timeline|score|syntaxhighlight|source|pre|code)[^>]*>.*?</\1>", "", t, flags=re.S | re.I)
    t = re.sub(r"<[^>]+>", "", t)                                          # 남은 HTML 태그
    t = re.sub(r"\[\[(파일|File|그림|Image|분류|Category):[^\[\]]*(\[\[[^\]]*\]\][^\[\]]*)*\]\]", "", t, flags=re.I)
    t = re.sub(r"\[\[([^\]|]*)\|([^\]]*)\]\]", r"\2", t)                   # [[문서|보이는 글]] → 보이는 글
    t = re.sub(r"\[\[([^\]]*)\]\]", r"\1", t)                              # [[문서]] → 문서
    t = re.sub(r"\[https?://[^\s\]]+ ([^\]]*)\]", r"\1", t)                # [주소 글] → 글
    t = re.sub(r"\[https?://[^\]]*\]", "", t)
    t = re.sub(r"'{2,}", "", t)                                            # 굵게·기울임
    t = re.sub(r"__[A-Z]+__", "", t)
    out = []
    for line in t.split("\n"):
        line = line.strip()
        if not line or line.startswith(("=", "|", "!", "{", "}")):          # 제목, 표의 나머지
            continue
        line = re.sub(r"^[*#:;]+\s*", "", line)                            # 목록 기호
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            out.append(line)
    return out

def sentences(paragraph):
    return [s.strip() for s in re.split(r"(?<=[.!?。])\s+", paragraph) if s.strip()]

hangul = lambda s: sum("가" <= ch <= "힣" for ch in s)
keep = lambda p: len(p) >= 20 and hangul(p) / max(1, sum(ch.isalpha() for ch in p)) >= 0.5
docs = [" ".join(p for p in clean(text) if keep(p)) for _, text in pages]
held = set(random.Random(0).sample(range(len(pages)), len(pages) // 10))   # 4·5장과 같은 검증용 문서
train_docs = [d for i, d in enumerate(docs) if i not in held and d]
val_docs = [d for i, d in enumerate(docs) if i in held and d]
print(len(train_docs), len(val_docs), sum(map(len, train_docs)), sum(map(len, val_docs)))

In [ ]:
pick = random.Random(0)
paras = [p for d in train_docs for p in d.split(". ")]
letters = {ch for ch, n in Counter(ch for p in pick.sample(paras, 300000) for ch in p).items() if n >= 5}
UNK_CH, SP = "□", "▁"
to_text = lambda t: "".join(ch if ch in letters else UNK_CH for ch in t).replace(" ", SP)
tok = Tokenizer(models.BPE(unk_token=UNK_CH))
tok.pre_tokenizer = pre_tokenizers.Split(SP, behavior="merged_with_next")
tok.train_from_iterator([to_text(p) for p in pick.sample(paras, 200000)], trainers.BpeTrainer(
    vocab_size=8000, show_progress=False, initial_alphabet=sorted(letters - {" "} | {SP, UNK_CH}),
    special_tokens=["<패딩>", "<시작>", "<끝>"]))
PAD, START, END, VOCAB = 0, 1, 2, tok.get_vocab_size()

LENGTH = 512                                               # 모델이 한 번에 읽는 토큰 수
def chunks(texts):
    """문서마다 <시작>·<끝>을 붙여 한 줄로 이은 다음, LENGTH + 1개씩 자른다(마지막 1개는 정답으로만 쓴다)."""
    stream = [t for e in tok.encode_batch([to_text(x) for x in texts]) for t in [START] + e.ids + [END]]
    n = len(stream) // (LENGTH + 1)
    return torch.tensor(stream[:n * (LENGTH + 1)]).view(n, LENGTH + 1)

train_rows = chunks(pick.sample(train_docs, len(train_docs)))
val_rows = chunks(val_docs)
show = lambda ids: tok.decode([i for i in ids if i > END]).replace(" ", "").replace(SP, " ")   # 토큰 번호를 글로
print(VOCAB, len(letters), tuple(train_rows.shape), tuple(val_rows.shape))
print(show(val_rows[0, :40].tolist()))

In [ ]:
def rotate(x, pos):
    """3장 3.2절의 RoPE. 이번에는 위치 번호 pos(배치, 길이)를 받아, 그 위치의 각도만큼 돌린다."""
    hd = x.shape[-1]
    theta = 10000.0 ** (-torch.arange(0, hd, 2).float() / hd)
    angle = pos[:, None, :, None].float() * theta                     # (배치, 1, 길이, hd/2)
    c, s = torch.cos(angle), torch.sin(angle)
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack([a * c - b * s, a * s + b * c], dim=-1).flatten(-2)

class Attention(nn.Module):
    """RoPE를 넣은 셀프 어텐션. kv_heads가 heads보다 작으면 그룹 질의 어텐션(3장 3.5절)이다."""
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.heads, self.kv_heads, self.hd = heads, kv_heads, dim // heads
        self.to_q = nn.Linear(dim, dim)
        self.to_k = nn.Linear(dim, kv_heads * self.hd)
        self.to_v = nn.Linear(dim, kv_heads * self.hd)
        self.out = nn.Linear(dim, dim)

    def forward(self, x, pos, allow, cache=None):
        B, L, D = x.shape
        q = rotate(self.to_q(x).view(B, L, self.heads, self.hd).transpose(1, 2), pos)
        k = rotate(self.to_k(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2), pos)
        v = self.to_v(x).view(B, L, self.kv_heads, self.hd).transpose(1, 2)
        if cache is not None:                                 # 6.2절: 앞에서 만든 키·값 뒤에 이번 것을 붙여 저장한다
            if "k" in cache:
                k, v = torch.cat([cache["k"], k], dim=2), torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=allow, enable_gqa=self.kv_heads < self.heads)
        return self.out(a.transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, dim, heads, kv_heads):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attend = Attention(dim, heads, kv_heads)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x, pos, allow, cache=None):
        x = x + self.attend(self.norm1(x), pos, allow, cache)
        return x + self.mlp(self.norm2(x))

class LanguageModel(nn.Module):
    def __init__(self, dim=192, layers=4, heads=6, kv_heads=6, size=8000):
        super().__init__()
        self.table = nn.Embedding(size, dim)                  # RoPE를 쓰므로 위치 표가 없다
        self.blocks = nn.ModuleList(Block(dim, heads, kv_heads) for _ in range(layers))
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, size)

    def forward(self, ids, pos=None, caches=None, real=None):
        """ids(배치, 길이). caches를 주면 블록마다 키·값을 쌓는다. real은 패딩이 아닌 토큰(6.5절)."""
        B, L = ids.shape
        before = caches[0]["real"].shape[1] if caches and "real" in caches[0] else 0   # 캐시에 이미 있는 토큰 수
        if pos is None:
            pos = torch.arange(before, before + L).expand(B, L)
        if real is None:
            real = torch.ones(B, L, dtype=torch.bool)
        if caches is not None:
            real = torch.cat([caches[0]["real"], real], dim=1) if before else real
            caches[0]["real"] = real
        q_at = torch.arange(before, before + L)                # 질의와 키가 줄에서 몇 번째 칸인지
        k_at = torch.arange(before + L)
        # 앞 칸만 본다(인과 마스크). 패딩 키는 보지 않되, 패딩 칸의 질의는 자기 자신만은 보게 해서 계산이 끊기지 않게 한다
        allow = (k_at[None, :] <= q_at[:, None]) & (real[:, None, :] | (k_at[None, :] == q_at[:, None]))
        x = self.table(ids)
        for i, block in enumerate(self.blocks):
            x = block(x, pos, allow[:, None], caches[i] if caches is not None else None)
        return self.head(self.norm(x))

count = lambda net: sum(p.numel() for p in net.parameters())
for kv in [6, 2]:
    print(f"키·값 머리 {kv}: 파라미터 {count(LanguageModel(kv_heads=kv)):,}")

In [ ]:
def fit(net, X, lr=0.003, batch=16, seed=0):
    """5장과 같다: 1에폭, AdamW, 처음 10% 걸음 동안 올리고 코사인으로 줄인다."""
    optimizer = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.0)
    shuffle = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
    steps = math.ceil(len(X) / batch)
    warm = max(1, steps // 10)
    net.train()
    for step in range(1, steps + 1):
        ids = X[shuffle[(step - 1) * batch:step * batch]]
        for group in optimizer.param_groups:
            group["lr"] = lr * min(1, step / warm) * 0.5 * (1 + math.cos(math.pi * step / steps))
        out = net(ids[:, :-1])
        loss = F.cross_entropy(out.reshape(-1, VOCAB), ids[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    net.eval()
    return net

def loss_of(net, X):
    with torch.inference_mode():
        return sum(F.cross_entropy(net(X[k:k + 8, :-1]).reshape(-1, VOCAB), X[k:k + 8, 1:].reshape(-1), reduction="sum").item()
                   for k in range(0, len(X), 8)) / X[:, 1:].numel()

TRAIN = train_rows[:8_000_000 // LENGTH]                   # 토큰 800만 개(1에폭)
nets, sec = {}, {}
for kv in [6, 2]:                                           # 6.4절에서 쓸 GQA 모델(키·값 머리 2개)도 함께 학습시킨다
    torch.manual_seed(0)
    lap = time.perf_counter()
    nets[kv] = fit(LanguageModel(kv_heads=kv), TRAIN)
    sec[kv] = time.perf_counter() - lap
    print(f"키·값 머리 {kv}: 학습 {sec[kv] / 60:.1f}분, 검증 손실 {loss_of(nets[kv], val_rows[:200]):.4f}", flush=True)
net = nets[6]

In [ ]:
def generate_naive(net, prompt, n):
    """캐시 없이: 걸음마다 지금까지의 토큰 전체를 다시 넣고, 마지막 칸의 점수에서 가장 높은 토큰을 고른다."""
    ids, times, scores = prompt, [], []
    with torch.inference_mode():
        for _ in range(n):
            lap = time.perf_counter()
            last = net(ids)[:, -1]
            nxt = last.argmax(dim=-1, keepdim=True)
            times.append(time.perf_counter() - lap)
            scores.append(last)
            ids = torch.cat([ids, nxt], dim=1)
    return ids, times, torch.stack(scores, dim=1)

prompt = val_rows[1:2, :64]                                 # 검증용 문서의 한 줄에서 앞 64토큰
generate_naive(net, prompt, 4)                              # 시간을 재기 전에 한 번 돌려 둔다
out, t_naive, s_naive = generate_naive(net, prompt, LENGTH - 64)
print("앞 64토큰:", show(prompt[0].tolist()))
print("이어 만든 60토큰:", show(out[0, 64:124].tolist()))
print(f"첫 토큰까지 {t_naive[0] * 1000:.1f}ms, 둘째 토큰 {t_naive[1] * 1000:.1f}ms, 마지막(512번째 칸) {t_naive[-1] * 1000:.1f}ms, 합계 {sum(t_naive):.2f}초")

## 6.2 KV 캐시 직접 구현하기

In [ ]:
def generate_cached(net, prompt, n, forget_position=False):
    """KV 캐시로: 처음에 프롬프트 전체를 한 번 넣어 키·값을 쌓고(프리필), 그다음부터는 새 토큰 하나만 넣는다."""
    caches = [{} for _ in net.blocks]
    times, scores = [], []
    with torch.inference_mode():
        lap = time.perf_counter()
        last = net(prompt, caches=caches)[:, -1]              # 프리필
        for step in range(n):
            nxt = last.argmax(dim=-1, keepdim=True)
            times.append(time.perf_counter() - lap)
            scores.append(last)
            prompt = torch.cat([prompt, nxt], dim=1)
            if step == n - 1:
                break
            lap = time.perf_counter()
            # 잘못된 구현: 새 토큰 하나만 넣으면서 위치를 0부터 다시 센다(캐시에 쌓인 토큰 수를 더하지 않는다)
            pos = torch.zeros(nxt.shape, dtype=torch.long) if forget_position else None
            last = net(nxt, pos=pos, caches=caches)[:, -1]
    return prompt, times, torch.stack(scores, dim=1), caches

generate_cached(net, prompt, 4)
out_c, t_cached, s_cached, caches = generate_cached(net, prompt, LENGTH - 64)
print(f"만든 토큰이 같은 수: {(out_c == out).sum().item() - 64} / {LENGTH - 64}")
print(f"점수(로짓)의 차이: 최대 {(s_cached - s_naive).abs().max():.2e}")
print(f"첫 토큰까지 {t_cached[0] * 1000:.1f}ms, 둘째 토큰 {t_cached[1] * 1000:.1f}ms, 마지막 {t_cached[-1] * 1000:.1f}ms, 합계 {sum(t_cached):.2f}초")
print("캐시에 쌓인 키 하나의 모양:", tuple(caches[0]["k"].shape))

In [ ]:
# 검증용 문서 20줄에서 앞 64토큰씩을 프롬프트로 써서 64토큰씩 만들고, 세 가지를 비교한다
same, bug_same, worst, bug_first, diff_steps = 0, 0, 0.0, [], []
for r in range(20):
    p = val_rows[100 + r:101 + r, :64]
    a, _, sa = generate_naive(net, p, 64)
    b, _, sb, _ = generate_cached(net, p, 64)
    c, _, sc, _ = generate_cached(net, p, 64, forget_position=True)
    same += (a == b)[0, 64:].sum().item()
    bug_same += (a == c)[0, 64:].sum().item()
    worst = max(worst, (sa - sb).abs().max().item())
    wrong = (a != c)[0, 64:].nonzero()
    bug_first.append(wrong[0].item() + 1 if len(wrong) else 65)
    diff_steps.append([(sa - sb)[0, k].abs().max().item() for k in range(64)] + [(sa - sc)[0, k].abs().max().item() for k in range(64)])
print(f"캐시 없이와 같은 토큰: 올바른 캐시 {same} / 1280, 위치를 잊은 캐시 {bug_same} / 1280")
print(f"올바른 캐시의 로짓 차이 최대 {worst:.2e}")
print("위치를 잊은 캐시가 처음 다른 토큰을 낸 걸음:", bug_first)
print("예:", show(c[0, 64:104].tolist()))

In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter, NullFormatter
for path in (glob.glob(os.path.expanduser("~/Library/Fonts/NotoSansCJK*.otf"))
             + glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc")):
    font_manager.fontManager.addfont(path)
plt.rcParams.update({"font.family": ["Noto Sans CJK KR", "DejaVu Sans"], "font.size": 9, "axes.unicode_minus": False})
BLUE, ORANGE, GREEN, PURPLE, RED, GRAY = "#2F6FDB", "#F28C28", "#1AA37A", "#8A5CD1", "#E5484D", "#9AA3AF"

fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.9), dpi=260)
ax = axes[0]
at = np.arange(64, LENGTH)                                  # 그 걸음에서 모델이 보는 토큰 수(64~511)
ax.plot(at, np.array(t_naive) * 1000, color=RED, lw=1.0, label=f"캐시 없이(합계 {sum(t_naive):.2f}초)")
ax.plot(at, np.array(t_cached) * 1000, color=GREEN, lw=1.0, label=f"KV 캐시(합계 {sum(t_cached):.2f}초)")
ax.plot([64], [t_naive[0] * 1000], "o", color=RED, ms=5)
ax.plot([64], [t_cached[0] * 1000], "o", color=GREEN, ms=5)
ax.annotate("첫 토큰(프리필)", (64, t_cached[0] * 1000), xytext=(26, 40), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="-", color=GRAY, lw=0.8))
ax.set_xlabel("그 걸음에서 모델이 보는 토큰 수", fontsize=8.4); ax.set_ylabel("한 걸음에 걸린 시간(ms)", fontsize=8.4)
ax.set_title("① 앞 64토큰 뒤에 448토큰을 만들 때 걸음마다 걸린 시간", loc="left", fontsize=8.6, color="#374151")
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax = axes[1]
# 첫 토큰은 두 방법 모두 프롬프트 전체를 한 번에 넣어 계산이 같으므로(차이 0) 둘째 토큰부터 그린다
ax.plot(range(2, 65), np.median([d[1:64] for d in diff_steps], axis=0), color=GREEN, lw=1.4, label="올바른 캐시")
ax.plot(range(2, 65), np.median([d[65:] for d in diff_steps], axis=0), color=RED, lw=1.4, label="위치를 잊은 캐시(새 토큰의 위치를 0으로)")
ax.set_yscale("log"); ax.set_xlabel("만든 토큰의 순서", fontsize=8.4); ax.set_ylabel("캐시 없이 낸 로짓과의 차이(최대)", fontsize=8.4)
ax.set_title("② 프롬프트 20개의 중앙값. 캐시 없이 만든 결과와 얼마나 다른가", loc="left", fontsize=8.6, color="#374151")
ax.legend(frameon=False, fontsize=8)
for a in axes:
    a.tick_params(labelsize=8)
    a.grid(color="#eef0f3", lw=0.6); a.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_6_steps.png"); plt.close(fig)

## 6.3 첫 토큰까지의 시간과 토큰당 시간

In [ ]:
def median_time(fn, repeat=5):
    fn()                                                    # 한 번 돌려 둔다
    laps = []
    for _ in range(repeat):
        lap = time.perf_counter(); fn(); laps.append(time.perf_counter() - lap)
    return float(np.median(laps))

PROMPTS = [16, 32, 64, 128, 256, 448]
ttft, per_token = {}, {}
with torch.inference_mode():
    for P in PROMPTS:
        p = val_rows[2:3, :P]
        ttft[P] = median_time(lambda: net(p, caches=[{} for _ in net.blocks]))
        caches = [{} for _ in net.blocks]
        net(p, caches=caches)
        one = val_rows[2:3, P:P + 1]
        def step():                                         # 캐시 길이를 P로 둔 채 새 토큰 하나를 넣는다
            for c in caches:
                c["k"], c["v"] = c["k"][:, :, :P], c["v"][:, :, :P]
            caches[0]["real"] = caches[0]["real"][:, :P]
            net(one, caches=caches)
        per_token[P] = median_time(step)
        print(f"프롬프트 {P:>3}토큰: 첫 토큰까지 {ttft[P] * 1000:6.1f}ms  그 뒤 토큰 하나 {per_token[P] * 1000:5.2f}ms  "
              f"프롬프트 토큰 하나당 프리필 {ttft[P] / P * 1000:.3f}ms", flush=True)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.3), dpi=260)
ax.plot(PROMPTS, [ttft[P] * 1000 for P in PROMPTS], "-o", color=BLUE, ms=4, lw=1.5, label="첫 토큰까지(프롬프트 전체를 한 번에)")
ax.plot(PROMPTS, [per_token[P] * 1000 for P in PROMPTS], "-o", color=ORANGE, ms=4, lw=1.5, label="그 뒤 토큰 하나(새 토큰 하나만)")
ax.plot(PROMPTS, [per_token[P] * P * 1000 for P in PROMPTS], "--", color=GRAY, lw=1.2, label="토큰 하나씩 프롬프트 길이만큼 넣었다면")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(PROMPTS); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}")); ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.tick_params(labelsize=8)
ax.set_xlabel("프롬프트 길이(토큰, 로그 눈금)", fontsize=8.4); ax.set_ylabel("시간(ms, 로그 눈금)", fontsize=8.4)
ax.legend(frameon=False, fontsize=8)
ax.grid(color="#eef0f3", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_6_prefill.png"); plt.close(fig)

## 6.4 KV 캐시의 메모리와 그룹 질의 어텐션

In [ ]:
nbytes = lambda caches: sum(c[k].numel() * c[k].element_size() for c in caches for k in ("k", "v"))
param_bytes = {kv: sum(p.numel() * p.element_size() for p in nets[kv].parameters()) for kv in nets}
print(f"파라미터: 키·값 머리 6개 {param_bytes[6]:,}바이트, 2개 {param_bytes[2]:,}바이트")
rows = []
with torch.inference_mode():
    for kv in [6, 2]:
        for B, T in [(1, 512), (8, 512), (64, 512)]:
            caches = [{} for _ in nets[kv].blocks]
            nets[kv](val_rows[:B, :T], caches=caches)
            formula = 2 * len(nets[kv].blocks) * B * T * kv * 32 * 4   # 키와 값 × 블록 × 배치 × 길이 × 머리 × 머리 차원 × 4바이트
            rows.append((kv, B, T, nbytes(caches), formula))
            print(f"키·값 머리 {kv}  배치 {B:>2}  길이 {T}: 캐시 {nbytes(caches):>12,}바이트  식 {formula:>12,}  파라미터의 {nbytes(caches) / param_bytes[kv]:.2f}배")

In [ ]:
# 같은 프롬프트로 키·값 머리 6개와 2개 모델의 걸음당 시간을 잰다(배치 1과 32, 캐시 길이 448)
gqa_time = {}
with torch.inference_mode():
    for kv in [6, 2]:
        for B in [1, 32]:
            caches = [{} for _ in nets[kv].blocks]
            nets[kv](val_rows[:B, :448], caches=caches)
            one = val_rows[:B, 448:449]
            def step():
                for c in caches:
                    c["k"], c["v"] = c["k"][:, :, :448], c["v"][:, :, :448]
                caches[0]["real"] = caches[0]["real"][:, :448]
                nets[kv](one, caches=caches)
            gqa_time[kv, B] = median_time(step)
            print(f"키·값 머리 {kv}  배치 {B:>2}: 걸음 하나 {gqa_time[kv, B] * 1000:.2f}ms", flush=True)
print("검증 손실:", {kv: round(loss_of(nets[kv], val_rows[:200]), 4) for kv in nets})

In [ ]:
from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(5.2, 3.4), dpi=260)
labels, cache_mb, param_mb = [], [], []
for kv, B, T, got, _ in rows:
    labels.append(f"머리 {kv}\n배치 {B}")
    cache_mb.append(got / 2**20); param_mb.append(param_bytes[kv] / 2**20)
x = np.arange(len(rows))
ax.bar(x - 0.2, param_mb, 0.38, color=GRAY)
ax.bar(x + 0.2, cache_mb, 0.38, color=[ORANGE if kv == 6 else PURPLE for kv, *_ in rows])
for i, v in enumerate(cache_mb):
    ax.text(i + 0.2, v * 1.15, f"{v:.1f}", ha="center", fontsize=8)
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(top=max(cache_mb) * 2)
ax.tick_params(axis="y", labelsize=8)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_ylabel("메모리(MiB, 로그 눈금)", fontsize=8.4)
ax.legend([Patch(color=GRAY), Patch(color=ORANGE), Patch(color=PURPLE)],
          ["파라미터", "KV 캐시(키·값 머리 6개)", "KV 캐시(머리 2개, GQA)"],
          frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, handlelength=1.2, columnspacing=1.0)
ax.grid(axis="y", color="#eef0f3", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout(); fig.savefig("fig_6_memory.png"); plt.close(fig)

## 6.5 여러 문장을 묶어 만들기: 배치와 패딩

In [ ]:
BATCHES = [1, 2, 4, 8, 16, 32, 64]
batch_step, batch_ttft = {}, {}
with torch.inference_mode():
    for B in BATCHES:
        p = val_rows[:B, :64]
        batch_ttft[B] = median_time(lambda: net(p, caches=[{} for _ in net.blocks]), repeat=3)
        caches = [{} for _ in net.blocks]
        net(p, caches=caches)
        one = val_rows[:B, 64:65]
        def step():
            for c in caches:
                c["k"], c["v"] = c["k"][:, :, :64], c["v"][:, :, :64]
            caches[0]["real"] = caches[0]["real"][:, :64]
            net(one, caches=caches)
        batch_step[B] = median_time(step)
        print(f"배치 {B:>2}: 첫 토큰까지 {batch_ttft[B] * 1000:7.1f}ms  걸음 하나 {batch_step[B] * 1000:6.2f}ms  "
              f"초당 토큰 {B / batch_step[B]:8,.0f}", flush=True)

In [ ]:
# 길이가 다른 프롬프트 8개를 한 배치로 묶어 24토큰씩 만들고, 하나씩 만든 결과와 비교한다
sizes = [12, 20, 28, 36, 44, 52, 60, 64]
texts = [val_rows[200 + i, :n] for i, n in enumerate(sizes)]
alone = [generate_cached(net, t[None], 24)[0][0, len(t):] for t in texts]

def batched(texts, n, side):
    """side="right": 뒤에 패딩, 마스크 없이. "right_mask": 뒤에 패딩, 패딩을 가림. "left": 앞에 패딩, 패딩을 가림."""
    W = max(map(len, texts))
    ids = torch.full((len(texts), W), PAD)
    real = torch.zeros(len(texts), W, dtype=torch.bool)
    for i, t in enumerate(texts):
        if side.startswith("right"):
            ids[i, :len(t)] = t; real[i, :len(t)] = True
        else:
            ids[i, W - len(t):] = t; real[i, W - len(t):] = True
    if side == "right":
        real[:] = True                                       # 패딩도 진짜 토큰처럼 본다
    caches = [{} for _ in net.blocks]
    made = []
    with torch.inference_mode():
        last = net(ids, caches=caches, real=real)[:, -1]
        for _ in range(n):
            nxt = last.argmax(dim=-1, keepdim=True)
            made.append(nxt)
            last = net(nxt, caches=caches)[:, -1]
    return ids, real, torch.cat(made, dim=1)

result = {}
for side in ["right", "right_mask", "left"]:
    ids, real, made = batched(texts, 24, side)
    result[side] = (ids, real, made)
    match = [(made[i] == alone[i]).float().mean().item() for i in range(len(texts))]
    print(side, "하나씩 만든 것과 같은 토큰 비율:", [round(m, 2) for m in match], f"평균 {np.mean(match):.3f}")
# 왼쪽 패딩에서 위치를 0부터 다시 세면(패딩 수만큼 빼면) 결과가 달라지는가. RoPE는 상대 위치만 쓰므로 같아야 한다
ids, real, _ = result["left"]
shifted = (real.cumsum(dim=1) - 1).clamp(min=0)
with torch.inference_mode():
    a = net(ids, real=real)[:, -1]
    b = net(ids, pos=shifted, real=real)[:, -1]
print(f"왼쪽 패딩, 위치를 밀지 않음과 민 것의 마지막 로짓 차이 {(a - b).abs().max():.2e}")
print("예(가장 짧은 프롬프트):", show(texts[0].tolist()), "→ 오른쪽 패딩:", show(result["right"][2][0, :12].tolist()),
      "/ 왼쪽 패딩:", show(result["left"][2][0, :12].tolist()))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5.2, 5.7), dpi=260, gridspec_kw={"height_ratios": [1, 1.05]})
ax = axes[0]
ax.plot(BATCHES, [B / batch_step[B] for B in BATCHES], "-o", color=GREEN, ms=4.5, lw=1.6, label="초당 만드는 토큰 수(배치 전체)")
ax.set_xscale("log", base=2); ax.set_xticks(BATCHES); ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xlabel("배치 크기(로그 눈금)", fontsize=8.4); ax.set_ylabel("초당 토큰", color=GREEN, fontsize=8.4)
ax2 = ax.twinx()
ax2.plot(BATCHES, [batch_step[B] * 1000 for B in BATCHES], "-s", color=ORANGE, ms=4, lw=1.4, label="걸음 하나의 시간(문장마다 기다리는 시간)")
ax2.set_ylabel("걸음 하나(ms)", color=ORANGE, fontsize=8.4)
ax.tick_params(labelsize=8); ax2.tick_params(labelsize=8)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, frameon=False, fontsize=8, loc="upper left")
ax.set_title("① 프롬프트 64토큰을 배치로 묶어 새 토큰을 만들 때", loc="left", fontsize=8.6, color="#374151")
ax.grid(color="#eef0f3", lw=0.6); ax.spines[["top"]].set_visible(False); ax2.spines[["top"]].set_visible(False)
ax = axes[1]
for row, (side, name) in enumerate([("right", "오른쪽 패딩, 마스크 없음"), ("left", "왼쪽 패딩, 패딩을 가림")]):
    ids, real, made = result[side]
    for i in range(4):                                       # 앞의 네 문장만 그린다
        y = row * 5 + i
        W = ids.shape[1]
        for j in range(W):
            pad = (ids[i, j] == PAD).item()
            ax.add_patch(plt.Rectangle((j, -y), 0.9, 0.8, color="#E5E7EB" if pad else "#B9CCF3", lw=0))
        for j in range(8):
            ok = (made[i, j] == alone[i][j]).item()
            ax.add_patch(plt.Rectangle((W + j, -y), 0.9, 0.8, color=GREEN if ok else RED, lw=0))
    ax.text(-1, -(row * 5) + 1.3, name, fontsize=8, ha="left", color="#374151")
ax.set_xlim(-1.5, ids.shape[1] + 9); ax.set_ylim(-8.4, 2.3); ax.axis("off")
ax.set_title("② 길이가 다른 네 문장. 회색 패딩, 파랑 프롬프트,\n만든 토큰 8개는 하나씩 만든 것과 같으면 초록, 다르면 빨강",
             loc="left", fontsize=8.6, color="#374151")
fig.tight_layout(); fig.savefig("fig_6_batch.png"); plt.close(fig)